In [1]:
pip install requests beautifulsoup4 pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import time
import re
import pandas as pd
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

BASE = 'http://www.ufcstats.com'

def get_driver():
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

def get_soup(driver, url, wait_time=5):
    driver.get(url)
    time.sleep(wait_time)
    return BeautifulSoup(driver.page_source, 'html.parser')

# ── Step 1: Get events from 2022+ ────────────────────────────
def get_events(driver):
    soup   = get_soup(driver, f'{BASE}/statistics/events/completed?page=all', wait_time=6)
    events = []

    for row in soup.select('tr'):
        link = row.select_one('a[href*="event-details"]')
        if not link:
            continue

        # Get all text from the row and search for a date pattern
        row_text = row.get_text(' ', strip=True)
        date_match = re.search(r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}', row_text)
        if not date_match:
            continue

        try:
            date = datetime.strptime(date_match.group(), '%B %d, %Y')
        except:
            continue

        if date.year >= 2022:
            events.append({
                'name': link.text.strip(),
                'url':  link['href'],
                'date': date,
            })

    # Remove duplicates
    seen = set()
    unique = []
    for e in events:
        if e['url'] not in seen:
            seen.add(e['url'])
            unique.append(e)

    print(f"Found {len(unique)} events from 2022+")
    if unique:
        print(f"  Most recent: {unique[0]['name']} ({unique[0]['date'].date()})")
        print(f"  Oldest:      {unique[-1]['name']} ({unique[-1]['date'].date()})")
    return unique

# ── Step 2: Get fight URLs from event ────────────────────────
def get_fights(driver, event_url):
    soup   = get_soup(driver, event_url, wait_time=5)
    fights = []

    for row in soup.select('tr.b-fight-details__table-row'):
        link = row.get('data-link', '')
        if link and 'fight-details' in link:
            fights.append(link)

    if not fights:
        for a in soup.select('a[href*="fight-details"]'):
            href = a.get('href','')
            if href not in fights:
                fights.append(href)

    return fights

# ── Step 3: Get fight result and fighter URLs ─────────────────
def get_fight_details(driver, fight_url):
    soup = get_soup(driver, fight_url, wait_time=5)

    names = [n.text.strip() for n in soup.select('a.b-fight-details__person-link')]
    if len(names) < 2:
        return None

    winner_els = soup.select('i.b-fight-details__person-status')
    if len(winner_els) < 2:
        return None

    winner = 'red' if winner_els[0].text.strip() == 'W' else 'blue'

    fighter_links = soup.select('a.b-fight-details__person-link')
    if len(fighter_links) < 2:
        return None

    return {
        'red_name':  names[0],
        'blue_name': names[1],
        'winner':    winner,
        'red_url':   fighter_links[0]['href'],
        'blue_url':  fighter_links[1]['href'],
    }

# ── Step 4: Get fighter career stats ─────────────────────────
fighter_cache = {}

def get_fighter_stats(driver, fighter_url):
    if fighter_url in fighter_cache:
        return fighter_cache[fighter_url]

    soup  = get_soup(driver, fighter_url, wait_time=4)
    stats = {}

    for item in soup.select('li.b-list__box-list-item'):
        text = item.text.strip()
        if ':' in text:
            key, val = text.split(':', 1)
            stats[key.strip()] = val.strip()

    # Win streak from fight history
    win_streak = 0
    for row in soup.select('tr.b-fight-details__table-row'):
        cols = row.select('td.b-fight-details__table-col')
        if cols:
            result = cols[0].text.strip().lower()
            if result == 'win':
                win_streak += 1
            elif result in ['loss', 'nc', 'draw']:
                break

    def parse_pct(val):
        try:    return float(val.replace('%','').replace('--','').strip()) / 100
        except: return None

    def parse_float(val):
        try:    return float(val.replace('--','').strip())
        except: return None

    def parse_height(val):
        try:
            parts = val.replace('"','').split("'")
            return round((int(parts[0].strip()) * 12 + int(parts[1].strip())) * 2.54, 1)
        except: return None

    def parse_reach(val):
        try:    return round(float(val.replace('"','').strip()) * 2.54, 1)
        except: return None

    def parse_age(dob_str):
        try:
            dob = datetime.strptime(dob_str, '%b %d, %Y')
            return round((datetime.today() - dob).days / 365.25, 1)
        except: return None

    def parse_ctrl(val):
        try:
            parts = val.split(':')
            return int(parts[0]) * 60 + int(parts[1])
        except: return None

    result = {
        'height':      parse_height(stats.get('Height',    '')),
        'reach':       parse_reach( stats.get('Reach',     '')),
        'age':         parse_age(   stats.get('DOB',       '')),
        'sig_str_acc': parse_pct(   stats.get('Str. Acc.', '')),
        'kd':          parse_float( stats.get('KD',        '')),
        'td_acc':      parse_pct(   stats.get('TD Acc.',   '')),
        'sub_att':     parse_float( stats.get('Sub. Avg.', '')),
        'ctrl_time':   parse_ctrl(  stats.get('Ctrl',      '')),
        'win_streak':  win_streak,
    }

    fighter_cache[fighter_url] = result
    return result

# ── Step 5: Main pipeline ─────────────────────────────────────
def main():
    driver     = get_driver()
    all_fights = []

    try:
        events = get_events(driver)

        for i, event in enumerate(events):
            print(f"[{i+1}/{len(events)}] {event['name']} ({event['date'].date()})")

            try:
                fight_urls = get_fights(driver, event['url'])
                print(f"  {len(fight_urls)} fights")
            except Exception as e:
                print(f"  Error getting fights: {e}")
                continue

            for fight_url in fight_urls:
                try:
                    fight = get_fight_details(driver, fight_url)
                    if not fight:
                        continue

                    red  = get_fighter_stats(driver, fight['red_url'])
                    blue = get_fighter_stats(driver, fight['blue_url'])

                    def diff(a, b):
                        if a is None or b is None: return None
                        return round(a - b, 4)

                    all_fights.append({
                        'event':           event['name'],
                        'date':            event['date'].date(),
                        'red_name':        fight['red_name'],
                        'blue_name':       fight['blue_name'],
                        'winner':          fight['winner'],
                        'reach_diff':      diff(red['reach'],       blue['reach']),
                        'height_diff':     diff(red['height'],      blue['height']),
                        'age_diff':        diff(red['age'],         blue['age']),
                        'sig_str_diff':    diff(red['sig_str_acc'], blue['sig_str_acc']),
                        'kd_diff':         diff(red['kd'],          blue['kd']),
                        'td_acc_diff':     diff(red['td_acc'],      blue['td_acc']),
                        'sub_att_diff':    diff(red['sub_att'],     blue['sub_att']),
                        'win_streak_diff': diff(red['win_streak'],  blue['win_streak']),
                        'ctrl_time_diff':  diff(red['ctrl_time'],   blue['ctrl_time']),
                        'red_age':         red['age'],
                        'blue_age':        blue['age'],
                        'red_win_streak':  red['win_streak'],
                        'blue_win_streak': blue['win_streak'],
                    })

                except Exception as e:
                    print(f"    Fight error: {e}")
                    continue

            # Checkpoint every 10 events
            if (i + 1) % 10 == 0:
                pd.DataFrame(all_fights).to_csv('ufc_2022_2025_preprocessed.csv', index=False)
                print(f"  Checkpoint — {len(all_fights)} fights saved")

            time.sleep(0.3)

    finally:
        driver.quit()

    df = pd.DataFrame(all_fights)
    df.to_csv('ufc_2022_2025_preprocessed.csv', index=False)
    print(f"\nDone. {len(df)} fights saved.")
    print(df.head())

if __name__ == '__main__':
    main()

Found 192 events from 2022+
  Most recent: UFC 329: McGregor vs. Holloway 2 (2026-07-11)
  Oldest:      UFC Fight Night: Kattar vs. Chikadze (2022-01-15)
[1/192] UFC 329: McGregor vs. Holloway 2 (2026-07-11)
  14 fights
[2/192] UFC Fight Night: Fiziev vs. Torres (2026-06-27)
  13 fights
[3/192] UFC Fight Night: Kape vs. Horiguchi (2026-06-20)
  12 fights
[4/192] UFC Freedom 250 (2026-06-14)
  7 fights
[5/192] UFC Fight Night: Muhammad vs. Bonfim (2026-06-06)
  12 fights
[6/192] UFC Fight Night: Song vs. Figueiredo (2026-05-30)
  13 fights
[7/192] UFC Fight Night: Allen vs. Costa (2026-05-16)
  13 fights
[8/192] UFC 328: Chimaev vs. Strickland (2026-05-09)
  13 fights
[9/192] UFC Fight Night: Della Maddalena vs. Prates (2026-05-02)
  13 fights
[10/192] UFC Fight Night: Sterling vs. Zalal (2026-04-25)
  13 fights
  Checkpoint — 123 fights saved
[11/192] UFC Fight Night: Burns vs. Malott (2026-04-18)
  12 fights
[12/192] UFC 327: Prochazka vs. Ulberg (2026-04-11)
  12 fights
[13/192] UFC 

In [8]:
import pandas as pd

# Load the scraped data — adjust path to where your file is
df_scraped = pd.read_csv('ufc_2022_2025_preprocessed.csv')

print(f"Shape: {df_scraped.shape}")
print(f"Date range: {df_scraped['date'].min()} to {df_scraped['date'].max()}")
print(f"Null counts:\n{df_scraped.isnull().sum()}")

Shape: (2346, 18)
Date range: 2022-01-15 to 2026-07-11
Null counts:
event                 0
date                  0
red_name              0
blue_name             0
winner                0
reach_diff            2
height_diff           2
age_diff              0
sig_str_diff          0
kd_diff            2346
td_acc_diff           0
sub_att_diff          0
win_streak_diff       0
ctrl_time_diff     2346
red_age               0
blue_age              0
red_win_streak        0
blue_win_streak       0
dtype: int64


In [1]:
import time
import pandas as pd
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

# ── Setup ─────────────────────────────────────────────────────
def get_driver():
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)')
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

def get_soup(driver, url, wait_time=4):
    driver.get(url)
    time.sleep(wait_time)
    return BeautifulSoup(driver.page_source, 'html.parser')

# ── Debug single fighter to verify parsing ────────────────────
def debug_fighter(driver, url):
    soup  = get_soup(driver, url, wait_time=5)
    stats = {}
    for item in soup.select('li.b-list__box-list-item'):
        text = item.text.strip()
        if ':' in text:
            key, val = text.split(':', 1)
            stats[key.strip()] = val.strip()
    print("All stats found on page:")
    for k, v in stats.items():
        print(f"  '{k}': '{v}'")
    return stats

# ── Parse KD and Ctrl ─────────────────────────────────────────
def parse_kd(val):
    try:
        return float(val.replace('--','').strip())
    except:
        return None

def parse_ctrl(val):
    # Stored as MM:SS per 15 min average e.g. "2:30"
    try:
        val = val.replace('--','').strip()
        if ':' in val:
            parts = val.split(':')
            return int(parts[0]) * 60 + int(parts[1])
        return float(val)
    except:
        return None

# ── Get KD and Ctrl for a fighter URL ────────────────────────
fighter_cache = {}

def get_kd_ctrl(driver, fighter_url):
    if fighter_url in fighter_cache:
        return fighter_cache[fighter_url]

    soup  = get_soup(driver, fighter_url, wait_time=4)
    stats = {}
    for item in soup.select('li.b-list__box-list-item'):
        text = item.text.strip()
        if ':' in text:
            key, val = text.split(':', 1)
            stats[key.strip()] = val.strip()

    result = {
        'kd':        parse_kd(stats.get('KD', '--')),
        'ctrl_time': parse_ctrl(stats.get('Ctrl', '--')),
    }
    fighter_cache[fighter_url] = result
    return result

# ── Main — patch existing CSV ─────────────────────────────────
def main():
    # Load existing scraped data
    df = pd.read_csv('ufc_2022_2025_preprocessed.csv')
    df = df.replace('None', None)

    print(f"Loaded {len(df)} fights")
    print(f"Current kd_diff nulls:       {df['kd_diff'].isna().sum()}")
    print(f"Current ctrl_time_diff nulls: {df['ctrl_time_diff'].isna().sum()}")

    driver = get_driver()

    # ── Debug first fighter to verify parsing ─────────────────
    print("\nDebugging first event to verify KD and Ctrl parsing...")
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    import re

    BASE = 'http://www.ufcstats.com'

    # Get first event
    soup = get_soup(driver, f'{BASE}/statistics/events/completed?page=all', wait_time=6)
    events = []
    for row in soup.select('tr'):
        link = row.select_one('a[href*="event-details"]')
        if not link:
            continue
        row_text = row.get_text(' ', strip=True)
        date_match = re.search(
            r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}',
            row_text
        )
        if not date_match:
            continue
        try:
            date = datetime.strptime(date_match.group(), '%B %d, %Y')
            if date.year >= 2022:
                events.append({'name': link.text.strip(), 'url': link['href'], 'date': date})
        except:
            continue

    seen, unique_events = set(), []
    for e in events:
        if e['url'] not in seen:
            seen.add(e['url'])
            unique_events.append(e)

    print(f"Found {len(unique_events)} events")

    # Get first fight from first event to debug
    first_event = unique_events[0]
    event_soup  = get_soup(driver, first_event['url'], wait_time=5)
    fight_links = []
    for row in event_soup.select('tr.b-fight-details__table-row'):
        link = row.get('data-link','')
        if link and 'fight-details' in link:
            fight_links.append(link)
    if not fight_links:
        for a in event_soup.select('a[href*="fight-details"]'):
            fight_links.append(a['href'])

    if fight_links:
        fight_soup    = get_soup(driver, fight_links[0], wait_time=5)
        fighter_links = fight_soup.select('a.b-fight-details__person-link')
        if fighter_links:
            print(f"\nDebugging fighter: {fighter_links[0].text.strip()}")
            stats = debug_fighter(driver, fighter_links[0]['href'])
            print(f"\nKD parsed:   {parse_kd(stats.get('KD','--'))}")
            print(f"Ctrl parsed: {parse_ctrl(stats.get('Ctrl','--'))}")

    driver.quit()

if __name__ == '__main__':
    main()

Loaded 2346 fights
Current kd_diff nulls:       2346
Current ctrl_time_diff nulls: 2346

Debugging first event to verify KD and Ctrl parsing...
Found 192 events

Debugging fighter: Conor McGregor
All stats found on page:
  'Height': '5' 9"'
  'Weight': '155 lbs.'
  'Reach': '74"'
  'STANCE': 'Southpaw'
  'DOB': 'Jul 14, 1988'
  'SLpM': '5.32'
  'Str. Acc.': '49%'
  'SApM': '4.66'
  'Str. Def': '54%'
  'TD Avg.': '0.67'
  'TD Acc.': '55%'
  'TD Def.': '66%'
  'Sub. Avg.': '0.1'

KD parsed:   None
Ctrl parsed: None


In [1]:
import pandas as pd
import numpy as np

# Load our scraped dataset
oos = pd.read_csv('ufc_2022_2025_preprocessed.csv')
oos = oos.replace('None', np.nan)

# Convert feature cols to numeric
feature_cols_7 = [
    'reach_diff', 'height_diff', 'age_diff', 'sig_str_diff',
    'td_acc_diff', 'sub_att_diff', 'win_streak_diff'
]
for col in feature_cols_7:
    oos[col] = pd.to_numeric(oos[col], errors='coerce')

# Search for tonight's fighters
fighters = [
    'Conor McGregor', 'Max Holloway',
    'Benoit Saint Denis', 'Paddy Pimblett',
    'Cory Sandhagen', 'Mario Bautista',
    'Brandon Royval', 'Lone\'er Kavanagh',
    'King Green', 'Terrance McKinney'
]

print("Searching for tonight's fighters in OOS dataset:\n")
for name in fighters:
    found = oos[
        (oos['red_name'].str.contains(name.split()[0], case=False, na=False)) |
        (oos['blue_name'].str.contains(name.split()[0], case=False, na=False))
    ]
    print(f"  {name}: {len(found)} fights found")

Searching for tonight's fighters in OOS dataset:

  Conor McGregor: 1 fights found
  Max Holloway: 21 fights found
  Benoit Saint Denis: 12 fights found
  Paddy Pimblett: 8 fights found
  Cory Sandhagen: 12 fights found
  Mario Bautista: 14 fights found
  Brandon Royval: 19 fights found
  Lone'er Kavanagh: 5 fights found
  King Green: 14 fights found
  Terrance McKinney: 14 fights found
